In [ ]:
import pandas as pd
import numpy as np
import re

# Load dataframe
df = pd.read_csv("found_ships_july.csv")

# Parse timestamps where possible (keep only time-of-day)
ts_parsed = pd.to_datetime(df["# Timestamp"], errors="coerce", infer_datetime_format=True)

# Build seconds-since-midnight series (float; will convert to int seconds)
seconds = pd.Series(np.nan, index=df.index, dtype=float)
mask_parsed = ts_parsed.notna()
if mask_parsed.any():
    parsed = ts_parsed[mask_parsed]
    seconds.loc[mask_parsed] = (
        parsed.dt.hour.astype(float) * 3600.0
        + parsed.dt.minute.astype(float) * 60.0
        + parsed.dt.second.astype(float)
        + parsed.dt.microsecond.astype(float) / 1e6
    )

# Fallback: extract HH:MM[:SS] with regex for unparsed rows
time_re = re.compile(r"(\d{1,2}):(\d{2})(?::(\d{2}))?")
for idx in seconds[seconds.isna()].index:
    txt = str(df.at[idx, "# Timestamp"]) if "# Timestamp" in df.columns else ""
    m = time_re.search(txt)
    if m:
        h = int(m.group(1))
        mn = int(m.group(2))
        s = int(m.group(3)) if m.group(3) else 0
        seconds.at[idx] = h * 3600 + mn * 60 + s

# Convert to integer second-of-day for exact matching
sec_int = pd.Series(seconds.round().astype("Int64"), index=df.index)

# Define exact intervals (inclusive bounds)
intervals = [
    (12 * 3600 + 18 * 60, 12 * 3600 + 26 * 60),   # 12:18-12:26
    (12 * 3600 + 38 * 60, 13 * 3600 + 0 * 60),    # 12:38-13:00
    (13 * 3600 + 33 * 60, 13 * 3600 + 59 * 60),   # 13:33-13:59
]

in_any_interval = pd.Series(False, index=df.index)
for lo, hi in intervals:
    in_any_interval |= (sec_int.notna() & (sec_int >= lo) & (sec_int <= hi))

filtered = df.loc[in_any_interval].copy()
filtered["__sec_of_day"] = sec_int[in_any_interval]

# Build canonical time-of-day string HH:MM:SS for deduplication
def fmt_hms(s):
    h = int(s // 3600)
    rem = int(s % 3600)
    m = rem // 60
    sec = rem % 60
    return f"{h:02d}:{m:02d}:{sec:02d}"

filtered["__time_of_day"] = filtered["__sec_of_day"].apply(lambda s: fmt_hms(s) if pd.notna(s) else None)

# Deduplicate: keep one row per exact second-level timestamp
filtered_unique = filtered.dropna(subset=["__time_of_day"]).drop_duplicates(subset=["__time_of_day"], keep="first")

# Summary per interval
print("Interval summaries (unique timestamps):")
for lo, hi in intervals:
    mask_i = (filtered_unique["__sec_of_day"] >= lo) & (filtered_unique["__sec_of_day"] <= hi)
    count_i = int(mask_i.sum())
    lo_str = fmt_hms(lo)
    hi_str = fmt_hms(hi)
    print(f"  {lo_str} - {hi_str}: {count_i} rows")

print(f"Total unique timestamps: {len(filtered_unique)}")

# Save results
filtered_unique.to_csv("specific_ships_july.csv", index=False)

filtered_unique

Interval summaries (unique timestamps):
  00:00:00 - 00:20:00: 40 rows
Total unique timestamps: 40


C:\Users\45422\AppData\Local\Temp\ipykernel_16588\3933893124.py:9: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  ts_parsed = pd.to_datetime(df["# Timestamp"], errors="coerce", infer_datetime_format=True)
C:\Users\45422\AppData\Local\Temp\ipykernel_16588\3933893124.py:9: UserWarning: Parsing dates in %d/%m/%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  ts_parsed = pd.to_datetime(df["# Timestamp"], errors="coerce", infer_datetime_format=True)


,# Timestamp,Type of mobile,MMSI,Latitude,Longitude,Navigational status,ROT,SOG,COG,Heading,...,Draught,Destination,ETA,Data source type,A,B,C,D,__sec_of_day,__time_of_day
0,29/06/2025 00:00:04,Class A,247389200,55.346797,11.041168,Under way using engine,0.0,9.0,359.2,355.0,...,NaN,Unknown,NaN,AIS,NaN,NaN,NaN,NaN,4,00:00:04
4,29/06/2025 00:00:14,Class A,247389200,55.347213,11.041163,Under way using engine,0.0,9.0,359.5,355.0,...,NaN,Unknown,NaN,AIS,NaN,NaN,NaN,NaN,14,00:00:14
7,29/06/2025 00:00:24,Class A,247389200,55.347630,11.041157,Under way using engine,0.0,9.0,359.3,355.0,...,NaN,Unknown,NaN,AIS,NaN,NaN,NaN,NaN,24,00:00:24
12,29/06/2025 00:00:33,Class A,247389200,55.348005,11.041152,Under way using engine,1.1,9.0,359.5,355.0,...,NaN,Unknown,NaN,AIS,NaN,NaN,NaN,NaN,33,00:00:33
17,29/06/2025 00:00:43,Class A,247389200,55.348422,11.041148,Under way using engine,0.0,9.0,359.5,355.0,...,NaN,Unknown,NaN,AIS,NaN,NaN,NaN,NaN,43,00:00:43
21,29/06/2025 00:00:55,Class A,247389200,55.348920,11.041145,Under way using engine,-1.1,9.0,359.7,355.0,...,NaN,Unknown,NaN,AIS,NaN,NaN,NaN,NaN,55,00:00:55
26,29/06/2025 00:05:43,AtoN,992191536,55.340950,11.028100,Unknown value,NaN,NaN,NaN,NaN,...,NaN,Unknown,NaN,AIS,1.0,1.0,1.0,1.0,343,00:05:43
35,29/06/2025 00:05:47,AtoN,992191537,55.342850,11.042867,Unknown value,NaN,NaN,NaN,NaN,...,NaN,Unknown,NaN,AIS,1.0,1.0,1.0,1.0,347,00:05:47
40,29/06/2025 00:11:44,AtoN,992191536,55.340950,11.028100,Unknown value,NaN,NaN,NaN,NaN,...,NaN,Unknown,NaN,AIS,1.0,1.0,1.0,1.0,704,00:11:44
44,29/06/2025 00:11:48,AtoN,992191537,55.342850,11.042867,Unknown value,NaN,NaN,NaN,NaN,...,NaN,Unknown,NaN,AIS,1.0,1.0,1.0,1.0,708,00:11:48
